In [ ]:
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots

In [ ]:
def create_folder(path):
    try:
        os.makedirs(path, exist_ok=True)
        print(f"Folder '{path}' created successfully.")
    except OSError as error:
        print(f"Folder already exists '{path}': {error}")

In [ ]:
save_figure_path = '' # Location to save figures here

In [ ]:
create_folder(save_figure_path)

In [ ]:
data = pd.read_csv("") # File path here

In [ ]:
data = data.drop('Title.1', axis=1)

# Year of publication

In [ ]:
data['year'] = data['Study ID'].str[-4:].astype(int)

In [ ]:
# Count the occurrences of each year
year_counts = data['year'].value_counts().sort_index().reset_index()
year_counts.columns = ['year', 'count']

# Create a complete range of years
all_years = pd.DataFrame({'year': range(year_counts['year'].min(), year_counts['year'].max() + 1)})

# Merge with the counts, filling missing years with 0
year_counts_complete = all_years.merge(year_counts, on='year', how='left').fillna(0)

# Country of Institutional Affiliation

In [ ]:
countries = data.copy(deep=True)
countries = countries[['Country/ies']]

In [ ]:
countries['Country/ies'] = countries['Country/ies'].str.split(',')
countries = countries.explode('Country/ies').reset_index()
countries['Country/ies'] = countries['Country/ies'].str.strip()

In [ ]:
countries['Country/ies'] = countries['Country/ies'].replace({'US': 'USA', 'United States': 'USA'})

In [ ]:
countries['Country/ies'].value_counts()

In [ ]:
countries_counts = countries['Country/ies'].value_counts().reset_index()
#countries_counts.rename(columns={'index': 'country', 'Country/ies': 'count'}, inplace=True)
countries_counts.rename(columns={'Country/ies': 'country'}, inplace=True)

### Get ISO Codes for Countries

In [ ]:
country_code_dict = {
    'USA': 'USA',
    'UK': 'GBR',
    'France': 'FRA',
    'Italy': 'ITA',
    'Germany': 'DEU',
    'Netherlands': 'NLD',
    'Switzerland': 'CH',
    'India': 'IND',
    'Belgium': 'BEL',
    'China': 'CHN',
    'Austria': 'AUT',
    'Canada': 'CAN', 
    'Turkey': 'TUR',
    'Sweden': 'SWE',
    'Australia': 'AUS',
    'Spain': 'ESP', 
    'Singapore': 'SGP', 
    'Norway': 'NOR', 
    'New Zealand': 'NZL', 
    'Malaysia': 'MYS', 
    'Ireland': 'IRL',
    'Denmark': 'DNK',
    'Republic of Korea': 'KOR',
    'Japan': 'JPN',
    'Czech Republic': 'CZE',
    'Iran': 'IRN', 
    'Monaco': 'MCO',
    'UAE': 'ARE',
    'Colombia': 'COL'
}

In [ ]:
countries_counts['iso_code'] = countries_counts['country'].map(country_code_dict)
countries_counts

In [ ]:
fig = px.choropleth(
    countries_counts,
    locations='iso_code',  # Column with ISO alpha-2 codes
    locationmode='ISO-3',      # Specify that we are using ISO-3 codes
    color='count',              # Column with values to color the countries
    hover_name='country',  # Hover text
    color_continuous_scale=px.colors.sequential.Plasma,  # Color scale
    title='Count of Articles by Country'
)

# Update layout for better presentation
fig.update_geos(
    showcoastlines=True,
    coastlinecolor='Black',
    showland=True,
    landcolor='LightGray'
)

# Show the plot
fig.show()

In [ ]:
# Prepare data for the histogram
year_counts = data['year'].value_counts().sort_index().reset_index()
year_counts.columns = ['year', 'count']
all_years = pd.DataFrame({'year': range(year_counts['year'].min(), year_counts['year'].max() + 1)})
year_counts_complete = all_years.merge(year_counts, on='year', how='left').fillna(0)

# Area of clinical focus/specialty

In [ ]:
data['Area(s) of Clinical Focus'].value_counts()

In [ ]:
specialty_list = data['Area(s) of Clinical Focus'].tolist()

In [ ]:
def split_semicolons(input_list):
    for i in range(0, len(input_list), 1):
        if ';' in input_list[i]:
            separated_items = input_list.pop(i)
            separated_items = separated_items.split(';')
            # add values back to list as separate values
            input_list.extend(separated_items)
    # remove leading and ending whitespace
    input_list = [item.strip() for item in input_list]
    return input_list

In [ ]:
'''
specialty_list = split_semicolons(specialty_list)
for item in specialty_list:
    print(item)
'''

In [ ]:
data_specialty_by_year = data.copy(deep=True)
data_specialty_by_year = data_specialty_by_year[['year', 'Area(s) of Clinical Focus']]

In [ ]:
data_specialty_by_year['Area(s) of Clinical Focus'] = data_specialty_by_year['Area(s) of Clinical Focus'].str.split(';')
data_specialty_by_year = data_specialty_by_year.explode('Area(s) of Clinical Focus').reset_index()

In [ ]:
data_specialty_by_year['Area(s) of Clinical Focus'] = data_specialty_by_year['Area(s) of Clinical Focus'].str.strip()

In [ ]:
data_specialty_by_year['Area(s) of Clinical Focus'].value_counts()

In [ ]:
df_specialty_counts = data_specialty_by_year.groupby(['year', 'Area(s) of Clinical Focus']).size().reset_index(name='count')

In [ ]:
df_count = df_specialty_counts

# Pivot the data to have years as columns and categories as rows
df_pivot = df_count.pivot(index='Area(s) of Clinical Focus', columns='year', values='count').fillna(0)

In [ ]:
# Pivot the data to have years as columns and categories as rows
df_pivot = df_count.pivot(index='Area(s) of Clinical Focus', columns='year', values='count').fillna(0)

# Create a custom color palette
colors = pc.qualitative.Plotly + pc.qualitative.Set1 + pc.qualitative.Pastel + pc.qualitative.Dark2[:5]

In [ ]:
specialty_counts = data_specialty_by_year['Area(s) of Clinical Focus'].value_counts().reset_index()
specialty_counts.columns = ['Specialty', 'Count']

In [ ]:
specialty_counts

# Model purpose, model scale

In [ ]:
data['DT Purpose'].value_counts()

In [ ]:
dt_purpose_list = data['DT Purpose'].to_list()
#dt_purpose_list = split_semicolons(dt_purpose_list)


In [ ]:
# Create a new list to store the modified items
dt_purpose_list_cleaned = []

# Iterate through the original list
for item in dt_purpose_list:
    if ';' in item:
        # Split the item and extend the new list with the separated items
        separated_items = item.split(';')
        dt_purpose_list_cleaned.extend(separated_items)
    else:
        # If there's no semicolon, just add the item as is
        dt_purpose_list_cleaned.append(item)

# Strip whitespace from all items
dt_purpose_list_cleaned = [item.strip() for item in dt_purpose_list_cleaned]

In [ ]:
dt_purpose_list_cleaned = [item.replace('Predictive (real-world scenarios)', 'Predictive') for item in dt_purpose_list_cleaned]
dt_purpose_list_cleaned = [item.replace('Simulation (hypothetical scenarios)', 'Simulation') for item in dt_purpose_list_cleaned]

In [ ]:
dt_model_scale = data['Scale of DT'].value_counts()

In [ ]:
dt_scale_list = data['Scale of DT'].to_list()

In [ ]:
# removing nan values (floats) and leaving strings (actual categories)
dt_scale_list = [x for x in dt_scale_list if not (isinstance(x, float))]

In [ ]:
# Create a new list to store the modified items
dt_scale_list_cleaned = []

# Iterate through the original list
for item in dt_scale_list:
    if ';' in item:
        # Split the item and extend the new list with the separated items
        separated_items = item.split(';')
        dt_scale_list_cleaned.extend(separated_items)
    else:
        # If there's no semicolon, just add the item as is
        dt_scale_list_cleaned.append(item)

# Strip whitespace from all items
dt_scale_list_cleaned = [item.strip() for item in dt_scale_list_cleaned]

In [ ]:
dt_scale_list_cleaned = [item.replace('Molecular (gene, gene network, etc.)', 'Molecular') for item in dt_scale_list_cleaned]

# Model components, method(s) used to create the models, type(s) of data

In [ ]:
data['DT components'].value_counts()

In [ ]:
DT_components_counts = data['DT components'].value_counts().to_frame().reset_index()

DT_components_counts.columns = ['Components', 'Count']

In [ ]:
replace_dict = {'Physical twin (specific entity); Model': 'Physical twin + Digital twin', 
                'Physical twin (specific entity); Model; Data feed physical --> digital': 'Physical twin + Digital twin + One-way data feed (physical to digital)',
                'Model': 'Virtual model only',
                'Physical twin (specific entity); Model; Data feed physical --> digital; Data feed digital --> physical': 'Physical twin + Digital twin + Bidirectional data feed'}
DT_components_counts_cleaned = DT_components_counts.replace(replace_dict)

In [ ]:
data['Method(s) Used to Build DTs'].value_counts()

In [ ]:
DT_methods_counts = data['Method(s) Used to Build DTs'].value_counts().to_frame().reset_index()

DT_methods_counts.columns = ['Methods', 'Count']

In [ ]:
DT_methods_counts['Methods'] = DT_methods_counts['Methods'].str.replace('; ', ' + ')

In [ ]:
dt_methods_counts_list = DT_methods_counts['Methods'].tolist()


#### Separating out the methods

In [ ]:
dt_methods_list = data['Method(s) Used to Build DTs'].to_list()

In [ ]:
# Create a new list to store the modified items
dt_methods_list_cleaned = []
counter = 1
# Iterate through the original list

for item in dt_methods_list:
    #print(f'***ARTICLE #{counter}***')
    #print(f'{item}')
    if isinstance(item, str):
        if '; ' in item:
            #print(f'item: {item}')
            # Split the item and extend the new list with the separated items
            separated_items = item.split('; ')
            #print(f'adding the following items to list: {separated_items}')
            dt_methods_list_cleaned.extend(separated_items)
        else:
            # If there's no semicolon, just add the item as is
            #print(f'adding the following items to list: {item}')
            dt_methods_list_cleaned.append(item)
    counter += 1

# Strip whitespace from all items
dt_methods_list_cleaned = [item.strip() for item in dt_methods_list_cleaned]

# Data quality variables

## Evaluation/Validation Methods

In [ ]:
data

In [ ]:
model_eval_methods = data.loc[data['Model Evaluation Method(s)'].notna(), 'Model Evaluation Method(s)'].tolist()

In [ ]:
data['Model Evaluation Method(s)'].isna().sum()

In [ ]:
num_articles = len(data)
num_no_eval = data['Model Evaluation Method(s)'].isna().sum()
num_with_eval = len(model_eval_methods)

print(f'Total number of articles: {num_articles} - {num_no_eval} null = {num_articles - num_no_eval}')
print(f'Number of articles with an evaluation method: {num_with_eval}')

In [ ]:
'''
i = 1
for item in model_eval_methods:
    print(f'{i}. {item}')
    i += 1
'''

In [ ]:
# Create a new list with all lowercase to check certain methods
model_eval_methods_check_list = [word.casefold() for word in model_eval_methods]

In [ ]:
search_groups = {
    # common metrics
    "AUC": ["auc", "auroc"],
    "Accuracy": ["accuracy"],
    "Sensitivity": ["sensitivity"],
    "Specificity": ["specificity"],
    "Precision": ["precision"],
    "Recall": ["recall"],
    # statistical tests
    "Correlation": ["correlation"],
    "ANOVA": ["anova"],
    "Chi-square": ["chi square", "chi squared", "chi-square", "chi-squared"],
    "Kruskal-Wallis Test": ["kruskal-wallis"],
    "Fisher Exact Test": ["fisher"],
    # ml/dl metrics
    "RMSE": ["rmse", "root mean square error", "root mean squared error"],
    "MAE": ["mae", "mean absolute error"],
    "MAPE": ["mape", "mean absolute percentage error"],
    # clinical data & outcomes
    "Ground Truth Geometries": ["correspondence between a bi-atrial ssm and the clinical geometry", "overlap between model and actual", "accuracy in shape, positions and orientations"],
    "Time to Progression": ["time to progression", "ttp"],
    "Real Patient Data": ["comparison to clinical outcomes", "comparison with patient data", "comparison to actual outcomes", "patient-level validation", "clinicians’ outcomes"],
    # literature & existing models
    "Gold Standard": ["state of the art", "reference", "gold"],
    "Comparison with Literature": ["comparison to existing literature", "comparison with data from cohorts in other published articles", "comparison with another study", "modal analysis using an existing study"],
    # other
    "Pre/Post Comparison": ["pre-post", "pre/post"],
    "Comparison with Non-Clinical Data": ["comparison with manufacturer-tabulated data", "comparison to in vitro model"],
    "Comparison of Statistical Properties Between Data": ["comparison of statistical properties of data generated with existing data from a database"],
    "Expert Evaluation": ["expert evaluation"],
    "Separate Validation Set": ["tested prospectively against 101 new patients", "validation set"]


}

term_counts = {
    group_name: sum(
        any(term in item for term in terms)  # Check for ANY term in the group
        for item in model_eval_methods_check_list
    )
    for group_name, terms in search_groups.items()
}

print(term_counts)

In [ ]:
# Convert dictionary to DataFrame
term_counts_df = pd.DataFrame({"Validation/Evaluation Method": term_counts.keys(), "Number of Articles": term_counts.values()})

## Type of Data Used to Test Model

In [ ]:
# Fill null values with "Unsure"
data['DT tested against'] = data['DT tested against'].fillna('Unsure')

In [ ]:
# Create list from column values
dt_data_test_list = data['DT tested against'].to_list()

# Create a new list to store the modified items
dt_data_test_list_cleaned = []

# Iterate through the original list
for item in dt_data_test_list:
    if ';' in item:
        # Split the item and extend the new list with the separated items
        separated_items = item.split(';')
        dt_data_test_list_cleaned.extend(separated_items)
    else:
        # If there's no semicolon, just add the item as is
        dt_data_test_list_cleaned.append(item)

# Strip whitespace from all items
dt_data_test_list_cleaned = [item.strip() for item in dt_data_test_list_cleaned]

## Source of Data

In [ ]:
data

In [ ]:
data_sources = data.loc[data['Source(s) of Data'].notna(), 'Source(s) of Data'].tolist()

In [ ]:
'''
i = 1
for item in data_sources:
    print(f'{i}. {item}')
    i += 1
'''

In [ ]:
# Create a new list with all lowercase to check certain methods
data_sources_check_list = [word.casefold() for word in data_sources]

In [ ]:
search_groups = {
    # common metrics
    "Clinical Data & Imaging": ["clinical data", "clinical care", "clinical practice", "echocardiograms", "ecg", "x-ray", "ct", "mri", "measured airway p-v data"],
    "Synthetic Data": ["synthetic", "simulated"],
    "Data Repository": ["dataset", "repository", "database", "archive"],
    "Other Studies": ["studies", "study", "trial", "program's"],
    "In-vitro Models/Cell Models": ["stem cell–derived cardiomyocyte", "in vitro model"],
    "Validated Models/Relationships": ["validated biomechanical model", "known relationships between genes/gene networks"],
    "Sensors": ["sensors", "time series biosignal data", "wearable device data", "twin precision nutrition"] #Twin Precision Nutrition Program is the set of diabetes studies that used continuous glucose monitors
    


}

data_source_counts = {
    group_name: sum(
        any(term in item for term in terms)  # Check for ANY term in the group
        for item in data_sources_check_list
    )
    for group_name, terms in search_groups.items()
}

print(data_source_counts)

In [ ]:
# Convert dictionary to DataFrame for Plotly Express
data_source_counts_df = pd.DataFrame({"Source of Data": data_source_counts.keys(), "Number of Articles": data_source_counts.values()})

# Number of Participants

In [ ]:
number_of_participants = data.loc[data['Number of Participants Providing Data (Sample)'].notna(), 'Number of Participants Providing Data (Sample)'].tolist()

In [ ]:
# Create a new list to store the modified items
number_of_participants_cleaned = []

# Iterate through the original list
for item in number_of_participants:
    if ';' in item:
        # Split the item and extend the new list with the separated items
        separated_items = item.split(';')
        number_of_participants_cleaned.extend(separated_items)
    elif '(validation),' in item:
        # Split the item and extend the new list with the separated items
        separated_items = item.split('(validation),')
        number_of_participants_cleaned.extend(separated_items)
    else:
        # If there's no semicolon, just add the item as is
        number_of_participants_cleaned.append(item)

# Strip whitespace from all items
number_of_participants_cleaned = [item.replace("calibration", "") for item in number_of_participants_cleaned]
number_of_participants_cleaned = [item.replace("()", "") for item in number_of_participants_cleaned]
number_of_participants_cleaned = [item.strip() for item in number_of_participants_cleaned]

In [ ]:
number_of_participants_ints = []  
for s in number_of_participants_cleaned:  
    try:  
        number_of_participants_ints.append(int(s))  
    except ValueError:  
        print(f"Skipping invalid value: {s}")

In [ ]:
# Create DataFrame
number_of_participants_df = pd.DataFrame({'Number of Participants': number_of_participants_ints})